# 03. Model Development Lab
**Objective:** Train, tune, and evaluate machine learning models (LightGBM/XGBoost) for alpha generation.

### Steps:
1. **Feature Selection:** Filter low-variance or highly correlated features.
2. **Training:** Train model on historical window.
3. **Evaluation:** Check Out-of-Sample (OOS) IC and Feature Importance.

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path('../').resolve()
sys.path.append(str(PROJECT_ROOT))

from config.settings import config
from quant_alpha.utils import load_parquet
from quant_alpha.models.lightgbm_model import LightGBMModel
from scripts.train_models import safe_winsorize, safe_sector_neutral_normalize, weighted_symmetric_mae

%matplotlib inline

## 1. Data Prep & Split

In [ ]:
df = load_parquet(config.CACHE_DIR / 'master_data_with_factors.parquet')
df['date'] = pd.to_datetime(df['date'])

# FIX: Calculate target if missing
if 'target' not in df.columns:
    print("⚠️ 'target' column missing. Calculating Sector-Neutral Return...")
    if 'raw_ret_5d' not in df.columns:
        if 'open' in df.columns:
            print("⚠️ 'raw_ret_5d' missing. Calculating from Open prices...")
            df = df.sort_values(['ticker', 'date'])
            df['next_open'] = df.groupby('ticker')['open'].shift(-1)
            df['future_open'] = df.groupby('ticker')['open'].shift(-6)
            df['raw_ret_5d'] = (df['future_open'] / df['next_open']) - 1
        else:
            print("❌ 'open' column missing. Cannot calculate returns.")

    if 'raw_ret_5d' in df.columns and 'sector' in df.columns:
        sector_mean = df.groupby(['date', 'sector'], observed=False)['raw_ret_5d'].transform('mean')
        df['target'] = df['raw_ret_5d'] - sector_mean
    elif 'raw_ret_5d' in df.columns:
        df['target'] = df['raw_ret_5d']
    else:
        print("❌ Could not create 'target' column.")

if 'target' in df.columns:
    df = df.dropna(subset=['target'])

# Define Features
exclude = ['date', 'ticker', 'target', 'raw_ret_5d', 'open', 'high', 'low', 'close', 'volume', 'sector', 'industry']
features = [c for c in df.columns if c not in exclude and np.issubdtype(df[c].dtype, np.number)]

# Preprocessing
print("Preprocessing...")
df = safe_winsorize(df, features)
df = safe_sector_neutral_normalize(df, features)

# Split (Train: < 2022, Test: >= 2022)
split_date = '2022-01-01'
train_df = df[df['date'] < split_date]
test_df = df[df['date'] >= split_date]
print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")

## 2. Train LightGBM Model

In [ ]:
params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 5,
    'objective': weighted_symmetric_mae,
    'n_jobs': -1
}

model = LightGBMModel(params=params)
model.fit(train_df[features], train_df['target'])

## 3. Evaluation (OOS IC)

In [ ]:
test_df['pred'] = model.predict(test_df[features])

ic = test_df.groupby('date').apply(lambda x: x['pred'].corr(x['target'], method='spearman'))
print(f"OOS Mean IC: {ic.mean():.4f}")
print(f"OOS IC IR:   {ic.mean() / ic.std():.4f}")

ic.rolling(20).mean().plot(title='Rolling 20d IC (OOS)', figsize=(10, 5))

## 4. Feature Importance

In [ ]:
imp = pd.Series(model.model.feature_importances_, index=features).sort_values(ascending=False).head(20)
imp.plot(kind='barh', figsize=(8, 8), title='Top 20 Features')